# Architectures génératives à base de tokens

Vincent Guigue (vincent.guigue@agroparistech.fr)

Ce TP vise à utiliser directement des RNN / LSTM et à travailler sur des **tokens** (morceaux de mots) pour générer du texte, comme les modèles de langue de l'état de l'art.

**Principe** (*language model*) : à chaque pas de temps, le réseau lit un token et prédit le **token suivant**. Une fois appris, on génère un texte en ré-injectant les tokens prédits.

Corpus : des discours de Donald Trump (≈ 17 000 phrases).

**Budget** : l'apprentissage prend de 10 à 30 minutes par modèle selon la machine. Des modèles pré-entraînés (et le tokenizer correspondant) sont fournis dans `models/` : avec `TRAIN = False`, on les charge directement.

## Table des matières

- [0. Mise en place : librairies, device, TensorBoard](#0-mise-en-place-librairies-device-tensorboard)
- [A. Chargement des données](#a-chargement-des-donnees)
    - [A.1 Construction du vocabulaire : le *tokenizer*](#a-1-construction-du-vocabulaire-le-tokenizer)
    - [A.2 Chaîne de traitement](#a-2-chaine-de-traitement)
- [B. Création du réseau](#b-creation-du-reseau)
- [C. La loss : cross-entropie masquée](#c-la-loss-cross-entropie-masquee)
- [D. Apprentissage](#d-apprentissage)
- [E. Génération de phrases et analyse des résultats](#e-generation-de-phrases-et-analyse-des-resultats)
    - [E.1 Génération gloutonne (*greedy*)](#e-1-generation-gloutonne-greedy)
    - [E.2 Génération stochastique](#e-2-generation-stochastique)
    - [E.3 Recherche par faisceau (*beam search*)](#e-3-recherche-par-faisceau-beam-search)

<a id="0-mise-en-place-librairies-device-tensorboard"></a>
## 0. Mise en place : librairies, device, TensorBoard

In [ ]:
import os, re, sys, time, math
from pathlib import Path
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

In [ ]:
# ---- Choix du device ----
# Contrairement aux TP précédents, le coût est ici dominé par la couche de sortie (softmax sur 10 000 tokens) :
# le GPU (MPS sur Mac, CUDA sinon) accélère nettement l'apprentissage (~3x mesuré sur un Mac M1).
# Mettre USE_MPS = False en cas de problème avec MPS.
USE_MPS = True

if torch.cuda.is_available():
    device = torch.device("cuda")
elif USE_MPS and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("torch :", torch.__version__, "| device :", device)
_ = torch.manual_seed(0)  # reproductibilité

In [ ]:
# ---- TensorBoard ----
from pathlib import Path
TB_PATH = Path("/tmp/logs/module4")    # une expérience = un sous-répertoire : SummaryWriter(TB_PATH / nom)
try:
    from torch.utils.tensorboard import SummaryWriter
except ImportError:                    # tensorboard absent : on continue sans enregistrer les courbes
    class SummaryWriter:
        def __init__(self, *args, **kwargs): pass
        def add_scalar(self, *args, **kwargs): pass
        def close(self): pass
    print("tensorboard non installé : les courbes ne seront pas enregistrées")

# Option 1 (recommandée) : dans un terminal, lancer la commande ci-dessous puis ouvrir http://localhost:6006
print(f"tensorboard --logdir {TB_PATH}")

# Option 2 : TensorBoard directement dans le notebook (décommenter les 2 lignes)
# %load_ext tensorboard
# %tensorboard --logdir {TB_PATH}

In [ ]:
# ---- Sauvegarde / chargement des modèles (sous-répertoire models/) ----
import os
os.makedirs("models", exist_ok=True)

def save_model(model, fichier):
    torch.save({"model_state": model.state_dict()}, fichier)

def load_model(fichier, model):
    if os.path.isfile(fichier):
        model.load_state_dict(torch.load(fichier, map_location="cpu")["model_state"])
    else:
        print(f"Erreur de chargement de {fichier} :\n => ATTENTION : initialisation aléatoire")
    return model.to(device)

<a id="a-chargement-des-donnees"></a>
## A. Chargement des données

Tout le code de cette partie est fourni.

In [ ]:
DATA_PATH = "./data/trump_full_speech.txt"

# OPERATION préliminaire : ouvrir le fichier pour comprendre la nature brute des données
alltxts = Path(DATA_PATH).read_text()
print(alltxts[:500])  # 500 premiers caractères

<a id="a-1-construction-du-vocabulaire-le-tokenizer"></a>
### A.1 Construction du vocabulaire : le *tokenizer*

On introduit un *tokenizer* pour découper le texte en tokens (mots fréquents, ou morceaux de mots pour les mots rares). Ce découpage est non trivial (cf. cours) : on utilise un algorithme *WordPiece* de la librairie [`tokenizers`](https://huggingface.co/docs/tokenizers) de HuggingFace, appris sur notre corpus.

**Attention** : l'apprentissage du tokenizer n'est pas déterministe (deux apprentissages donnent des numérotations de tokens différentes). Un modèle n'a de sens qu'avec **le tokenizer avec lequel il a été appris** : on le sauvegarde donc avec les modèles.

In [ ]:
from tokenizers import Tokenizer, decoders
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

TOKENIZER_FILE = "models/4_4_tokenizer.json"

if os.path.isfile(TOKENIZER_FILE):
    tokenizer = Tokenizer.from_file(TOKENIZER_FILE)
    print("tokenizer chargé depuis", TOKENIZER_FILE)
else:
    tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = Whitespace()       # pré-découpage sur les espaces et la ponctuation
    tokenizer.decoder = decoders.WordPiece()     # pour recoller les morceaux de mots (##) au décodage
    trainer = WordPieceTrainer(vocab_size=10000, # taille du vocabulaire
                               special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"])
    tokenizer.train([DATA_PATH], trainer)
    tokenizer.save(TOKENIZER_FILE)
    print("tokenizer appris et sauvegardé dans", TOKENIZER_FILE)
print("taille du vocabulaire :", tokenizer.get_vocab_size())

Exemples importants pour comprendre le principe de la tokenisation, les inégalités entre langues, les problèmes potentiels avec du vocabulaire technique...

In [ ]:
for s in ["This is an example.", "Exemple en francais, loin de Trump", "Immigration is tremendous."]:
    out = tokenizer.encode(s)
    print("Tokens:", out.tokens)
    print("IDs   :", out.ids)
    print("Décodage :", tokenizer.decode(out.ids), "\n")

In [ ]:
# indices des tokens spéciaux
PAD = tokenizer.token_to_id("[PAD]")
EOS = tokenizer.token_to_id(".")      # le point sert de fin de phrase (End Of Sentence)
print("PAD", PAD, "| EOS", EOS)

<a id="a-2-chaine-de-traitement"></a>
### A.2 Chaîne de traitement

1. Découpage du texte en phrases (sur les points), tokenisation, `Dataset`
2. `DataLoader` avec *padding* (séquences *temps en premier* : `longueur x batch`)

On se place dans une logique générative : pas d'ensemble de test (c'est discutable). Les phrases sont tronquées à `MAXLEN` tokens : quelques phrases très longues (jusqu'à 175 tokens) obligeraient à padder tout leur batch, ce qui multiplierait le temps de calcul.

In [ ]:
class TextDataset(Dataset):
    def __init__(self, text: str, tokenizer, maxsent=None, maxlen=None):
        """ Dataset de phrases tokenisées
            * text : texte brut
            * tokenizer : tokenizer (texte => [int])
            * maxsent : nombre maximum de phrases
            * maxlen : longueur maximale des phrases (en tokens)
        """
        phrases = [re.sub(' +', ' ', p).strip() for p in text.split(".")]
        self.phrases = [p + "." for p in phrases if len(p) > 0][:maxsent]
        self.phrasesnum = [torch.tensor(tokenizer.encode(p).ids[:maxlen]) for p in self.phrases]

    def __len__(self):
        return len(self.phrases)

    def __getitem__(self, i):
        return self.phrasesnum[i]


MAXLEN = 40
ds = TextDataset(alltxts, tokenizer, maxlen=MAXLEN)
print("nb phrases :", len(ds))
for i in range(3):
    print(f"doc {i} :", ds[i])
print(tokenizer.decode(ds[0].tolist()))
# savez-vous retrouver les tokens EOS (.) ?

In [ ]:
BATCH_SIZE = 64

def collate_fn(batch):
    lengths = torch.tensor([len(seq) for seq in batch])
    padded_sequences = pad_sequence(batch, batch_first=False, padding_value=PAD)   # Lmax x batch
    return padded_sequences, lengths

data = DataLoader(ds, collate_fn=collate_fn, batch_size=BATCH_SIZE, shuffle=True)

x, lengths = next(iter(data))
print("x:", x.size())
print("lengths:", lengths)

<a id="b-creation-du-reseau"></a>
## B. Création du réseau

Un réseau récurrent suivi d'un décodeur linéaire appliqué à **chaque** pas de temps (*many-to-many*) : à partir de l'état caché $h_t$, on prédit une distribution sur le vocabulaire pour le token $t+1$.

- l'entrée est une séquence d'indices de tokens (`T x B`) => `nn.Embedding` (appris cette fois) ;
- le `forward` renvoie les logits (`T x B x n_voc`) **et** l'état caché final, pour pouvoir reprendre la génération là où on s'est arrêté ;
- argument `cell` pour choisir entre `nn.RNN` et `nn.LSTM`. Rappel : l'état d'un LSTM est un couple `(h, c)`, mais il suffit de le transmettre tel quel au module.

In [ ]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, n_voc, cell="rnn"):
        super().__init__()
        self.emb = nn.Embedding(n_voc, input_size)
        # 1. Définition du module récurrent (self.rec), selon cell
        # 2. Définition du décodeur (self.decoder)
        # pas de piège, juste une réflexion sur les dimensions
        ##  TODO 

    def forward(self, input, state=None):
        """ input : T x B (indices) ; state : état caché initial (None => 0)
            => logits : T x B x n_voc, état caché final """
        ##  TODO 
        return output, state

In [ ]:
input_size, hidden_size = 128, 64
rnn = RNN(input_size, hidden_size, tokenizer.get_vocab_size(), cell="rnn")

# sanity check : est-ce que les données passent dans le réseau ?
x, lengths = next(iter(data))
output, state = rnn(x)
print("x:", x.size(), "=> output:", output.size())

<a id="c-la-loss-cross-entropie-masquee"></a>
## C. La loss : cross-entropie masquée

L'étape difficile maintenant :
1. calcul des $\hat y$ ;
2. mise en face des vérités terrain = **le token d'après** : on donne au réseau `x[:-1]` et la cible est `x[1:]` ;
3. ... mais il ne faut pas appliquer la loss sur le *padding* ;
4. ... et on veut conserver un traitement par batch => introduire un **masque** sur la loss : il faut raisonner en termes de graphe de calcul (les termes masqués ne doivent pas contribuer au gradient).

Indice : `nn.CrossEntropyLoss(reduction="none")` renvoie la loss de chaque exemple au lieu de leur moyenne.

In [ ]:
def maskedCrossEntropy(output: torch.Tensor, target: torch.LongTensor, padcar: int):
    """
    :param output: Tenseur length x batch x output_dim
    :param target: Tenseur length x batch
    :param padcar: index du token de padding
    """
    ##  TODO 

In [ ]:
x, lengths = next(iter(data))
x, y = x[:-1], x[1:]        # entrée / cible décalée d'un pas de temps
yhat, _ = rnn(x)
print("x:", x.size(), "| y:", y.size(), "| yhat:", yhat.size())
l = maskedCrossEntropy(yhat, y, PAD)
print("loss masquée :", l.item())
# vérification : PyTorch sait faire directement avec l'argument ignore_index
print("ignore_index :", nn.CrossEntropyLoss(ignore_index=PAD)(yhat.flatten(0, 1), y.flatten()).item())

<a id="d-apprentissage"></a>
## D. Apprentissage

Boucle standard sur le seul ensemble d'apprentissage. On suit la loss et la **perplexité** $= \exp(\text{loss})$ : c'est le nombre « effectif » de tokens entre lesquels le modèle hésite (10 000 pour un modèle aléatoire uniforme).

On apprend deux modèles : un RNN simple et un LSTM.

In [ ]:
def train(model, epochs, train_loader, name):
    writer = SummaryWriter(TB_PATH / f"{name}-{time.strftime('%Y%m%d-%H%M%S')}")
    model = model.to(device)
    optim = torch.optim.Adam(model.parameters(), lr=1e-3)
    for epoch in tqdm(range(epochs), desc=name):
        model.train()
        cumloss, count = 0., 0
        for x, lengths in train_loader:
            x = x.to(device)
            x, y = x[:-1], x[1:]
            optim.zero_grad()
            yhat, _ = model(x)
            l = maskedCrossEntropy(yhat, y, PAD)
            l.backward()
            optim.step()
            ntok = (y != PAD).sum().item()      # moyenne pondérée par le nombre de vrais tokens
            cumloss += l.item() * ntok
            count += ntok
        writer.add_scalar("loss/train", cumloss / count, epoch)
        writer.add_scalar("perplexity/train", math.exp(cumloss / count), epoch)
    writer.close()
    return model

In [ ]:
TRAIN = False   # False : on charge les modèles fournis dans models/ (qui vont avec le tokenizer sauvegardé)
EPOCHS = 40      # 15 à 45 s/epoch selon la machine (MPS / CPU)

models = {"rnn": RNN(input_size, hidden_size, tokenizer.get_vocab_size(), cell="rnn"),
          "lstm": RNN(input_size, hidden_size, tokenizer.get_vocab_size(), cell="lstm")}
for cell, model in models.items():
    fichier = f"models/4_4_{cell}.pt"
    if TRAIN:
        train(model, EPOCHS, data, f"trump-{cell}")
        save_model(model, fichier)
    else:
        load_model(fichier, model)
rnn, lstm = models["rnn"], models["lstm"]

<a id="e-generation-de-phrases-et-analyse-des-resultats"></a>
## E. Génération de phrases et analyse des résultats

1. On commence par la version naïve : à chaque pas, on choisit le token le plus probable (*greedy*) ;
2. puis une génération stochastique : on **tire** le token suivant selon la distribution prédite (avec une *température*) ;
3. enfin une recherche par faisceau (*beam search*).

<a id="e-1-generation-gloutonne-greedy"></a>
### E.1 Génération gloutonne (*greedy*)

Compléter la boucle. Version la plus simple : à chaque pas, on refait passer **toute** la séquence dans le réseau.

In [ ]:
def start_tensor(start, eos=EOS):
    """ texte de départ => tenseur T x 1 (si start est vide, on part d'une fin de phrase) """
    ids = tokenizer.encode(start).ids or [eos]
    return torch.tensor(ids, device=device).unsqueeze(1)

model = lstm.eval()
x = start_tensor("immigration")
with torch.no_grad():
    for i in range(30):   # longueur arbitraire
        # 1. Faire passer x dans le réseau
        # 2. Récupérer l'argmax de la prédiction au DERNIER pas de temps
        # 3. Concaténer l'indice à x
        ##  TODO 

# 4. Décoder la chaîne générée
print(tokenizer.decode(x.squeeze(1).tolist()))

**Questions** : que remarquez-vous sur les phrases générées (répétitions, boucles) ? Quel est le coût de calcul de cette version en fonction de la longueur générée ? Comment l'éviter grâce à l'état caché renvoyé par le `forward` ?

<a id="e-2-generation-stochastique"></a>
### E.2 Génération stochastique

Écrire une fonction de génération qui :
- lit le début de phrase `start` **une seule fois**, puis avance d'un token à la fois en ré-injectant l'état caché ;
- tire le token suivant selon la distribution prédite : `torch.multinomial(torch.softmax(logits / temperature, dim=-1), 1)` ;
- s'arrête quand le token `eos` est généré (ou après `maxlen` tokens).

Faire varier la température (0.5, 1, 1.5) : quel est son effet ?

In [ ]:
def generate(model, tokenizer, eos, start="", maxlen=50, temperature=1.0):
    """ Génération stochastique : initialise le réseau avec start (ou une fin de phrase si start est vide)
        et génère une séquence de longueur maximale maxlen ou qui s'arrête quand eos est généré. """
    model.eval()
    x = start_tensor(start, eos)
    seq = x.squeeze(1).tolist()
    with torch.no_grad():
        ##  TODO 
    return tokenizer.decode(seq)

for temperature in [0.5, 1.0, 1.5]:
    print(f"T={temperature} :", generate(lstm, tokenizer, EOS, start="Trump is", maxlen=30, temperature=temperature))

In [ ]:
##  TODO 

<a id="e-3-recherche-par-faisceau-beam-search"></a>
### E.3 Recherche par faisceau (*beam search*)

À chaque itération, on explore pour chacun des `k` candidats les `k` tokens les plus probables ; parmi les `k x k` séquences obtenues, on ne conserve que les `k` meilleures (au sens de la vraisemblance) pour l'itération suivante.

La log-vraisemblance d'une séquence (somme de log-probabilités négatives) diminue avec sa longueur : le beam search favorise donc les phrases courtes. On corrige en divisant le score par une **pénalité de longueur** ([normalisation OpenNMT](https://opennmt.net/OpenNMT/translation/beam_search/#length-normalization)) — uniquement pour *classer* les candidats, la log-probabilité cumulée reste, elle, non normalisée.

In [ ]:
class LengthPower():
    def __init__(self, base=5, alpha=0.7):
        self.base = base
        self.alpha = alpha

    def __call__(self, length):
        # https://opennmt.net/OpenNMT/translation/beam_search/#length-normalization
        return ((self.base + length) / (self.base + 1)) ** self.alpha

In [ ]:
def generate_beam(model, tokenizer, eos, k, start="", maxlen=30, lp=LengthPower()):
    """ Génère une séquence par beam search de largeur k. """
    model.eval()
    x = start_tensor(start, eos)
    n0 = len(x)
    with torch.no_grad():
        y, state = model(x)
        # un candidat = (log-proba cumulée, séquence, état caché, logits du token suivant, terminé ?)
        beams = [(0., x.squeeze(1).tolist(), state, y[-1, 0], False)]
        for _ in range(maxlen):
            ##  TODO 
    return tokenizer.decode(beams[0][1])

for name, model in [("RNN", rnn), ("LSTM", lstm)]:
    print(f"{name:5s} | greedy :", generate_beam(model, tokenizer, EOS, 1, start="Immigration"))
    print(f"{name:5s} | beam 5 :", generate_beam(model, tokenizer, EOS, 5, start="Immigration"))

**Pour aller plus loin** : comparer RNN et LSTM (perplexité dans TensorBoard, qualité des phrases) ; mettre de côté une partie des phrases pour mesurer la perplexité en test ; augmenter la taille du réseau ; varier la largeur du faisceau et le paramètre `alpha` de la pénalité de longueur.